# Reproduce Fig3 tumor state analysis

This notebook reconstructs the local `Cloud/civa_raw/Fig3_tumor_state_analysis` directory without using the former HPC output path.

Two input modes are supported:

1. **Rebuild mode** uses local `Flex.rds` and `lam.rds`, recreates the CIVA Tumor/PDO integration, and saves `combined_tumor.rds`.
2. **Saved-object mode** loads `Cloud/civa_raw/combined_tumor.rds` and reproduces downstream outputs.

Exact numerical equality with archived HPC results requires the same source objects, R/package versions, and random-number state. The archived QC had CIVA cluster counts 0 = 4 and 1 = 433; the current local RDS may differ.

## 1. Configure local paths and dependencies

In [ ]:
project_root <- Sys.getenv("CIVA_PROJECT_ROOT", unset = getwd())
project_root <- normalizePath(project_root, mustWork = TRUE)
if (basename(project_root) != "1_CIVA") {
  stop("Run this notebook from the 1_CIVA directory or set CIVA_PROJECT_ROOT.")
}

# Set TRUE only when local source objects are available and integration should be rebuilt.
rebuild_integration <- FALSE
run_expensive_gsea <- FALSE
random_seed <- 20260806L

flex_path <- Sys.getenv("CIVA_FLEX_RDS", unset = file.path(project_root, "Flex.rds"))
lam_path <- Sys.getenv("CIVA_LAM_RDS", unset = file.path(project_root, "lam.rds"))
combined_tumor_path <- file.path(project_root, "Cloud", "civa_raw", "combined_tumor.rds")
output_dir <- file.path(project_root, "Cloud", "civa_raw", "Fig3_tumor_state_analysis")
gsea_output_dir <- file.path(output_dir, "standard_GSEA")
dir.create(gsea_output_dir, recursive = TRUE, showWarnings = FALSE)

core_packages <- c("Seurat", "dplyr", "tidyr", "ggplot2", "patchwork", "Matrix")
rebuild_packages <- "harmony"
gsea_packages <- c("clusterProfiler", "enrichplot", "org.Hs.eg.db", "fgsea", "msigdbr")
missing_core <- core_packages[!vapply(core_packages, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing_core) > 0L) stop("Missing core R packages: ", paste(missing_core, collapse = ", "))
if (rebuild_integration && !requireNamespace("harmony", quietly = TRUE)) stop("Rebuild mode requires harmony.")
missing_gsea <- gsea_packages[!vapply(gsea_packages, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing_gsea) > 0L) {
  message("GSEA cells require: ", paste(missing_gsea, collapse = ", "))
  message("Install Bioconductor packages with BiocManager and msigdbr from CRAN, then restart R.")
}

suppressPackageStartupMessages({
  library(Seurat)
  library(dplyr)
  library(tidyr)
  library(ggplot2)
  library(patchwork)
  library(Matrix)
})
set.seed(random_seed)
list(project_root = project_root, flex_path = flex_path, lam_path = lam_path,
     combined_tumor_path = combined_tumor_path, output_dir = output_dir,
     rebuild_integration = rebuild_integration, run_expensive_gsea = run_expensive_gsea)

## 2-3. Load source objects and recreate integration

Rebuild mode reproduces the upstream integration from `Flex.rds` and `lam.rds`. Saved-object mode starts from the local derived RDS and is the practical option when the original HPC inputs are unavailable.

In [ ]:
required_metadata <- c("orig.ident", "seurat_clusters")

if (rebuild_integration) {
  missing_inputs <- c(flex_path, lam_path)[!file.exists(c(flex_path, lam_path))]
  if (length(missing_inputs) > 0L) {
    stop("Copy the HPC source RDS files locally or set CIVA_FLEX_RDS/CIVA_LAM_RDS. Missing: ",
         paste(missing_inputs, collapse = ", "))
  }
  flex <- readRDS(flex_path)
  lam <- readRDS(lam_path)
  if (!all(c("orig.ident", "cell_type") %in% colnames(flex[[]]))) {
    stop("Flex.rds must contain orig.ident and cell_type metadata.")
  }
  CIVA <- subset(flex, subset = orig.ident == "Assembloid" & cell_type == "Tumor")
  CIVA$orig.ident <- "CIVA Tumor"
  PDO <- lam
  PDO$orig.ident <- "PDO"
  combined_tumor <- merge(CIVA, y = PDO, add.cell.ids = c("CIVA_Tumor", "PDO"))
  if ("JoinLayers" %in% getNamespaceExports("SeuratObject")) combined_tumor <- JoinLayers(combined_tumor)
  combined_tumor <- NormalizeData(combined_tumor, verbose = FALSE)
  combined_tumor <- FindVariableFeatures(combined_tumor, verbose = FALSE)
  combined_tumor <- ScaleData(combined_tumor, features = VariableFeatures(combined_tumor), verbose = FALSE)
  combined_tumor <- RunPCA(combined_tumor, features = VariableFeatures(combined_tumor), npcs = 30, verbose = FALSE)
  combined_tumor <- harmony::RunHarmony(combined_tumor, group.by.vars = "orig.ident",
                                         reduction.use = "pca", dims.use = 1:30, verbose = FALSE)
  combined_tumor <- FindNeighbors(combined_tumor, reduction = "harmony", dims = 1:30, verbose = FALSE)
  combined_tumor <- FindClusters(combined_tumor, resolution = 0.1, verbose = FALSE)
  combined_tumor <- RunUMAP(combined_tumor, reduction = "harmony", dims = 1:30, verbose = FALSE)
  saveRDS(combined_tumor, combined_tumor_path)
} else {
  if (!file.exists(combined_tumor_path)) stop("Saved integration not found: ", combined_tumor_path)
  combined_tumor <- readRDS(combined_tumor_path)
}

metadata <- combined_tumor[[]]
if (!all(required_metadata %in% colnames(metadata))) {
  stop("combined_tumor is missing metadata: ", paste(setdiff(required_metadata, colnames(metadata)), collapse = ", "))
}
if (!"RNA" %in% Assays(combined_tumor)) stop("combined_tumor has no RNA assay.")
if (!all(c("CIVA Tumor", "PDO") %in% as.character(metadata$orig.ident))) {
  stop("orig.ident must contain both CIVA Tumor and PDO.")
}

current_counts <- table(as.character(metadata$orig.ident), as.character(metadata$seurat_clusters))
print(current_counts)
archived_counts <- c("CIVA Tumor|0" = 4L, "CIVA Tumor|1" = 433L, "PDO|0" = 1493L, "PDO|3" = 101L)
current_named <- setNames(as.integer(current_counts), paste(rownames(current_counts)[row(current_counts)], colnames(current_counts)[col(current_counts)], sep = "|"))
if (!all(current_named[names(archived_counts)] == archived_counts, na.rm = FALSE)) {
  warning("Current local cluster counts differ from the archived HPC outputs; regenerated values will not be identical.")
}

## 4-5. Identify immune-like cells and generate cancer-cell QC

In [ ]:
immune_qc_genes <- intersect(c("PTPRC", "LST1", "TYROBP", "AIF1", "C1QA", "C1QB", "CD68"), rownames(combined_tumor))
epithelial_qc_genes <- intersect(c("EPCAM", "TACSTD2", "KRT8", "KRT18", "KRT19", "KRT7"), rownames(combined_tumor))
if (length(immune_qc_genes) == 0L || length(epithelial_qc_genes) == 0L) stop("Identity QC marker panels do not overlap the RNA assay.")

identity_qc <- FetchData(combined_tumor, vars = c("orig.ident", "seurat_clusters", immune_qc_genes, epithelial_qc_genes), layer = "data")
identity_qc$immune_marker_mean <- rowMeans(identity_qc[, immune_qc_genes, drop = FALSE])
identity_qc$epithelial_marker_mean <- rowMeans(identity_qc[, epithelial_qc_genes, drop = FALSE])
identity_qc_summary <- identity_qc %>%
  mutate(orig.ident = as.character(orig.ident), seurat_clusters = as.character(seurat_clusters)) %>%
  group_by(orig.ident, seurat_clusters) %>%
  summarise(cells = n(), immune_marker_mean = mean(immune_marker_mean),
            epithelial_marker_mean = mean(epithelial_marker_mean),
            immune_marker_positive_percent = mean(immune_marker_mean > 0) * 100,
            epithelial_marker_positive_percent = mean(epithelial_marker_mean > 0) * 100,
            .groups = "drop") %>%
  arrange(orig.ident, seurat_clusters)
write.csv(identity_qc_summary, file.path(output_dir, "identity_qc_by_group_and_cluster.csv"), row.names = FALSE)
print(identity_qc_summary)

# Cluster 2 was validated as immune-like in the source analysis; exclude it from both groups.
macrophage_clusters <- "2"
keep_cells <- rownames(metadata)[as.character(metadata$orig.ident) %in% c("CIVA Tumor", "PDO") &
                                  !as.character(metadata$seurat_clusters) %in% macrophage_clusters]
corrected_cancer_cells <- subset(combined_tumor, cells = keep_cells)
corrected_cancer_cells$analysis_group <- factor(as.character(corrected_cancer_cells$orig.ident), levels = c("PDO", "CIVA Tumor"))
cancer_metadata <- corrected_cancer_cells[[]]
cancer_metadata$cell_id <- rownames(cancer_metadata)
cancer_metadata$analysis_group <- as.character(corrected_cancer_cells$analysis_group)

cancer_qc <- cancer_metadata %>% count(analysis_group, seurat_clusters, name = "cells") %>% arrange(analysis_group, seurat_clusters)
write.csv(cancer_qc, file.path(output_dir, "cancer_cell_qc.csv"), row.names = FALSE)
print(cancer_qc)

candidate_columns <- grep("sample|donor|patient|replicate|batch", colnames(cancer_metadata), value = TRUE, ignore.case = TRUE)
candidate_sample_summary <- bind_rows(lapply(candidate_columns, function(column) {
  sample_rows <- cancer_metadata %>%
    filter(!is.na(.data[[column]]), as.character(.data[[column]]) != "") %>%
    distinct(analysis_group, sample = .data[[column]])
  counts <- table(sample_rows$analysis_group)
  data.frame(metadata_column = column, groups = n_distinct(sample_rows$analysis_group),
             samples = n_distinct(sample_rows$sample),
             minimum_samples_per_group = if (length(counts) == 0L) 0L else min(counts))
}))
if (nrow(candidate_sample_summary) == 0L) {
  candidate_sample_summary <- data.frame(metadata_column = character(), groups = integer(),
                                         samples = integer(), minimum_samples_per_group = integer())
}
write.csv(candidate_sample_summary, file.path(output_dir, "candidate_sample_metadata.csv"), row.names = FALSE)

pseudobulk_sample_column <- NULL
pseudobulk_status <- data.frame(status = "not_run",
  detail = "No validated biological sample column was supplied; cell-level scores are descriptive only.")
if (!is.null(pseudobulk_sample_column)) {
  stop("Configure and validate a true sample column before implementing pseudobulk; orig.ident is not a replicate.")
}
write.csv(pseudobulk_status, file.path(output_dir, "pseudobulk_status.csv"), row.names = FALSE)
pseudobulk_status

## 6. Calculate tumor-state module scores

These Wilcoxon tests are exploratory cell-level comparisons, not biological-replicate inference.

In [ ]:
state_signatures <- list(
  IL1_response = c("NFKBIA", "IL1RN", "CXCL1", "CXCL2", "CXCL3", "CXCL8", "CCL2", "PTGS2", "TNFAIP3", "ICAM1"),
  TNF_NFKB = c("NFKBIA", "TNFAIP3", "TNF", "CCL2", "CXCL8", "ICAM1", "CSF1", "PTGS2"),
  Inflammatory_response = c("IL6", "CXCL8", "CCL2", "CSF2", "CXCL1", "CXCL2", "ICAM1", "PTGS2"),
  EMT = c("VIM", "FN1", "MMP2", "MMP9", "CDH1", "TNXB", "CCN1"),
  Hypoxia = c("HIF1A", "VEGFA", "LDHA", "SLC2A1", "CA9", "BNIP3", "DDIT4"),
  Candidate_regenerative_progenitor = c("KRT8", "KRT18", "KRT19", "KRT7", "EPCAM", "TACSTD2", "CLDN4", "KRT17", "LGALS3", "LRG1", "ORM1"),
  E2F_targets = c("MKI67", "TOP2A", "E2F2", "MCM2", "MCM4", "MCM6", "TYMS", "RAD51", "CDC6"),
  G2M_checkpoint = c("MKI67", "TOP2A", "CDK1", "CCNB1", "CDC20", "BUB1", "AURKB", "UBE2C")
)
state_signatures <- lapply(state_signatures, intersect, y = rownames(corrected_cancer_cells))
state_signatures <- state_signatures[lengths(state_signatures) >= 2L]
if (length(state_signatures) == 0L) stop("No state signature retains at least two genes.")

set.seed(random_seed)
corrected_cancer_cells <- AddModuleScore(corrected_cancer_cells, assay = "RNA", features = state_signatures,
                                         name = "state_score_", search = FALSE)
state_columns <- paste0("state_score_", seq_along(state_signatures))
score_long <- FetchData(corrected_cancer_cells, vars = c("analysis_group", state_columns)) %>%
  pivot_longer(cols = all_of(state_columns), names_to = "score_column", values_to = "score") %>%
  mutate(pathway = names(state_signatures)[match(score_column, state_columns)],
         analysis_group = factor(analysis_group, levels = c("PDO", "CIVA Tumor")))

pathway_score_summary <- score_long %>% group_by(pathway, analysis_group) %>%
  summarise(cells = n(), mean_score = mean(score), median_score = median(score), .groups = "drop")
pathway_score_tests <- score_long %>% group_by(pathway) %>%
  summarise(CIVA_Tumor_minus_PDO_median = median(score[analysis_group == "CIVA Tumor"]) - median(score[analysis_group == "PDO"]),
            cell_level_wilcox_p = wilcox.test(score ~ analysis_group, alternative = "two.sided")$p.value,
            .groups = "drop") %>%
  mutate(cell_level_FDR = p.adjust(cell_level_wilcox_p, method = "BH"),
         interpretation = "Corrected exploratory cell-level comparison; not a biological-replicate p-value.") %>%
  arrange(desc(abs(CIVA_Tumor_minus_PDO_median)))
write.csv(pathway_score_summary, file.path(output_dir, "cell_level_pathway_score_summary.csv"), row.names = FALSE)
write.csv(pathway_score_tests, file.path(output_dir, "cell_level_pathway_score_tests.csv"), row.names = FALSE)
print(pathway_score_tests)

options(repr.plot.width = 12, repr.plot.height = 7)
ggplot(score_long, aes(analysis_group, score, fill = analysis_group)) +
  geom_boxplot(outlier.shape = NA, width = 0.65) +
  geom_jitter(width = 0.12, alpha = 0.08, size = 0.25) +
  facet_wrap(~pathway, scales = "free_y", ncol = 4) +
  scale_fill_manual(values = c("PDO" = "#367A9A", "CIVA Tumor" = "#C66B3D")) +
  labs(title = "Cancer-cell state scores after immune-cluster exclusion", x = NULL, y = "Module score") +
  theme_classic(base_size = 12) + theme(legend.position = "none")

## 7. Run differential expression and standard GSEA

Set `run_expensive_gsea <- TRUE` in the configuration cell after installing the listed GSEA packages. Positive fold changes and NES values represent higher expression in CIVA Tumor.

In [ ]:
if (!run_expensive_gsea) {
  message("Skipping DEG/standard GSEA. Set run_expensive_gsea <- TRUE to regenerate these files.")
} else {
  missing_now <- gsea_packages[!vapply(gsea_packages, requireNamespace, logical(1), quietly = TRUE)]
  if (length(missing_now) > 0L) stop("Install GSEA packages first: ", paste(missing_now, collapse = ", "))

  gsea_cancer_cells <- corrected_cancer_cells
  Idents(gsea_cancer_cells) <- "analysis_group"
  deg_result <- FindMarkers(gsea_cancer_cells, ident.1 = "CIVA Tumor", ident.2 = "PDO",
                            min.pct = 0.10, logfc.threshold = 0, test.use = "wilcox", verbose = FALSE)
  deg_result$cluster <- "CIVA Tumor"
  deg_result$gene <- rownames(deg_result)
  civa_vs_pdo_deg <- deg_result %>% arrange(desc(avg_log2FC), p_val_adj)
  write.csv(civa_vs_pdo_deg, file.path(gsea_output_dir, "DEG_CIVA_Tumor_vs_PDO_FindAllMarkers.csv"), row.names = FALSE)

  ranked_degs <- civa_vs_pdo_deg %>% filter(!is.na(avg_log2FC), !duplicated(gene)) %>% arrange(desc(avg_log2FC))
  id_map <- clusterProfiler::bitr(ranked_degs$gene, fromType = "SYMBOL", toType = "ENTREZID", OrgDb = org.Hs.eg.db::org.Hs.eg.db) %>%
    distinct(SYMBOL, .keep_all = TRUE)
  ranked_entrez <- ranked_degs %>% inner_join(id_map, by = c("gene" = "SYMBOL")) %>%
    group_by(ENTREZID) %>% slice_max(abs(avg_log2FC), n = 1, with_ties = FALSE) %>% ungroup() %>% arrange(desc(avg_log2FC))
  gene_list <- ranked_entrez$avg_log2FC
  names(gene_list) <- ranked_entrez$ENTREZID
  gene_list <- sort(gene_list, decreasing = TRUE)
  if (length(gene_list) < 20L || length(unique(gene_list)) < 2L) stop("Ranked list is unsuitable for GSEA.")

  set.seed(random_seed)
  gsea_go_bp <- clusterProfiler::gseGO(geneList = gene_list, OrgDb = org.Hs.eg.db::org.Hs.eg.db,
    ont = "BP", keyType = "ENTREZID", minGSSize = 10, maxGSSize = 500,
    pvalueCutoff = 0.05, pAdjustMethod = "BH", verbose = FALSE)
  gsea_kegg <- clusterProfiler::gseKEGG(geneList = gene_list, organism = "hsa", minGSSize = 10,
    maxGSSize = 500, pvalueCutoff = 0.05, pAdjustMethod = "BH", verbose = FALSE)
  gsea_go_results <- as.data.frame(gsea_go_bp)
  gsea_kegg_results <- as.data.frame(gsea_kegg)
  write.csv(gsea_go_results, file.path(gsea_output_dir, "GSEA_GO_Biological_Process_CIVA_Tumor_vs_PDO.csv"), row.names = FALSE)
  write.csv(gsea_kegg_results, file.path(gsea_output_dir, "GSEA_KEGG_CIVA_Tumor_vs_PDO.csv"), row.names = FALSE)

  prioritized_pattern <- "interleukin|tumou?r necrosis factor|NF-kappaB|inflammatory|epithelial.*mesenchymal|hypoxia|regeneration|progenitor|E2F|G2M|cell cycle"
  prioritized_gsea_terms <- gsea_go_results %>% filter(grepl(prioritized_pattern, Description, ignore.case = TRUE)) %>% arrange(p.adjust)
  write.csv(prioritized_gsea_terms, file.path(gsea_output_dir, "GSEA_GO_prioritized_terms_CIVA_Tumor_vs_PDO.csv"), row.names = FALSE)

  if (nrow(gsea_go_results) > 0L) {
    go_plot <- enrichplot::dotplot(gsea_go_bp, showCategory = 20, title = "GO BP GSEA: CIVA Tumor versus PDO")
    ggsave(file.path(gsea_output_dir, "GSEA_GO_Biological_Process_dotplot.pdf"), go_plot, width = 11, height = 8)
  }
  if (nrow(prioritized_gsea_terms) > 0L) {
    top_go_plot <- enrichplot::gseaplot2(gsea_go_bp, geneSetID = prioritized_gsea_terms$ID[[1]],
                                         title = prioritized_gsea_terms$Description[[1]])
    ggsave(file.path(gsea_output_dir, "GSEA_top_prioritized_GO_term.pdf"), top_go_plot, width = 9, height = 6)
  }
  print(head(civa_vs_pdo_deg, 20))
}

## 8. Run focused MSigDB enrichment

In [ ]:
msigdbr_rows <- function(collection, subcollection = NULL) {
  args <- list(species = "Homo sapiens", collection = collection)
  if (!is.null(subcollection)) args$subcollection <- subcollection
  tryCatch(do.call(msigdbr::msigdbr, args), error = function(error) {
    legacy_args <- list(species = "Homo sapiens", category = collection)
    if (!is.null(subcollection)) legacy_args$subcategory <- subcollection
    do.call(msigdbr::msigdbr, legacy_args)
  })
}

if (run_expensive_gsea) {
  wanted_hallmarks <- c("HALLMARK_INFLAMMATORY_RESPONSE", "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION",
                        "HALLMARK_E2F_TARGETS", "HALLMARK_G2M_CHECKPOINT")
  wanted_go_bp <- c("GOBP_CYTOKINE_MEDIATED_SIGNALING_PATHWAY",
                    "GOBP_POSITIVE_REGULATION_OF_LEUKOCYTE_MIGRATION", "GOBP_MYELOID_LEUKOCYTE_MIGRATION")
  focused_rows <- bind_rows(msigdbr_rows("H") %>% filter(gs_name %in% wanted_hallmarks),
                            msigdbr_rows("C5", "GO:BP") %>% filter(gs_name %in% wanted_go_bp))
  focused_sets <- split(focused_rows$gene_symbol, focused_rows$gs_name)
  wanted_pathways <- c(wanted_hallmarks, wanted_go_bp)
  missing_sets <- setdiff(wanted_pathways, names(focused_sets))
  if (length(missing_sets) > 0L) stop("MSigDB sets not retrieved: ", paste(missing_sets, collapse = ", "))

  focused_gene_list <- ranked_degs$avg_log2FC
  names(focused_gene_list) <- ranked_degs$gene
  focused_gene_list <- sort(focused_gene_list, decreasing = TRUE)
  set.seed(random_seed)
  hallmark_gsea <- fgsea::fgseaMultilevel(pathways = focused_sets[wanted_pathways], stats = focused_gene_list,
                                           minSize = 10, maxSize = 500, eps = 0) %>% arrange(desc(NES))
  if (nrow(hallmark_gsea) == 0L) stop("No requested MSigDB genes overlapped the ranked list.")
  hallmark_export <- as.data.frame(hallmark_gsea)
  hallmark_export$leadingEdge <- vapply(hallmark_export$leadingEdge, paste, collapse = ";", FUN.VALUE = character(1))
  write.csv(hallmark_export, file.path(gsea_output_dir, "GSEA_MSigDB_Hallmark_CIVA_Tumor_vs_PDO.csv"), row.names = FALSE)
  print(hallmark_gsea)
} else {
  message("Skipping focused MSigDB GSEA.")
}

## 9. Generate publication-style GSEA figures

In [ ]:
save_both <- function(plot, stem, width = 10, height = 7) {
  ggsave(file.path(gsea_output_dir, paste0(stem, ".pdf")), plot, width = width, height = height)
  ggsave(file.path(gsea_output_dir, paste0(stem, ".png")), plot, width = width, height = height,
         dpi = 300, bg = "transparent")
}

if (run_expensive_gsea) {
  display_names <- c(
    HALLMARK_INFLAMMATORY_RESPONSE = "Inflammatory response",
    HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION = "Epithelial-mesenchymal transition",
    HALLMARK_E2F_TARGETS = "E2F targets",
    HALLMARK_G2M_CHECKPOINT = "G2/M checkpoint",
    GOBP_CYTOKINE_MEDIATED_SIGNALING_PATHWAY = "Cytokine-mediated signaling pathway",
    GOBP_POSITIVE_REGULATION_OF_LEUKOCYTE_MIGRATION = "Positive regulation of leukocyte migration",
    GOBP_MYELOID_LEUKOCYTE_MIGRATION = "Myeloid leukocyte migration"
  )
  plot_data <- as.data.frame(hallmark_gsea) %>%
    mutate(pathway_label = unname(display_names[pathway]),
           pathway_label = ifelse(is.na(pathway_label), gsub("_", " ", pathway), pathway_label),
           FDR = pmax(ifelse(is.na(padj), 1, padj), .Machine$double.xmin),
           nominal_p = pmax(ifelse(is.na(pval), 1, pval), .Machine$double.xmin),
           significance = factor(ifelse(FDR < 0.05, "FDR < 0.05", "FDR >= 0.05")),
           direction = factor(ifelse(NES >= 0, "CIVA Tumor-high", "PDO-high")),
           pathway_label = reorder(pathway_label, NES))
  base_theme <- theme_classic(base_size = 12) + theme(axis.text.y = element_text(color = "black"))

  signed_bar <- ggplot(plot_data, aes(NES, pathway_label, fill = direction)) +
    geom_col(width = 0.68) + geom_vline(xintercept = 0, linewidth = 0.4) +
    scale_fill_manual(values = c("CIVA Tumor-high" = "#C66B3D", "PDO-high" = "#367A9A")) +
    labs(title = "Focused MSigDB GSEA", x = "Normalized enrichment score", y = NULL, fill = NULL) + base_theme
  save_both(signed_bar, "GSEA_MSigDB_Hallmark_barplot", 9, 5.5)

  lollipop <- ggplot(plot_data, aes(NES, pathway_label, color = significance)) +
    geom_segment(aes(x = 0, xend = NES, yend = pathway_label), color = "grey65", linewidth = 0.7) +
    geom_point(size = 4) + geom_vline(xintercept = 0, linewidth = 0.4) +
    geom_text(aes(label = paste0("NES ", sprintf("%.2f", NES), ", FDR ", format.pval(FDR, digits = 2))),
              hjust = ifelse(plot_data$NES >= 0, -0.08, 1.08), size = 3) +
    scale_color_manual(values = c("FDR < 0.05" = "#B5452D", "FDR >= 0.05" = "grey55")) +
    scale_x_continuous(expand = expansion(mult = c(0.30, 0.30))) +
    labs(title = "Focused enrichment effect sizes", x = "Normalized enrichment score", y = NULL, color = NULL) + base_theme
  save_both(lollipop, "GSEA_MSigDB_Hallmark_lollipop", 11, 6)

  fdr_bar <- ggplot(plot_data, aes(NES, pathway_label, fill = -log10(FDR))) + geom_col(width = 0.65) +
    geom_vline(xintercept = 0, linewidth = 0.4) +
    scale_fill_gradient(low = "#DCEAF4", high = "#174A70", name = expression(-log[10](FDR))) +
    labs(title = "GSEA colored by FDR", x = "Normalized enrichment score", y = NULL) + base_theme
  save_both(fdr_bar, "GSEA_MSigDB_Hallmark_FDR_barplot")

  threshold_bar <- ggplot(plot_data, aes(NES, pathway_label, fill = significance)) + geom_col(width = 0.65) +
    geom_vline(xintercept = 0, linewidth = 0.4) +
    scale_fill_manual(values = c("FDR < 0.05" = "#174A70", "FDR >= 0.05" = "grey75")) +
    labs(title = "GSEA FDR threshold", x = "Normalized enrichment score", y = NULL, fill = NULL) + base_theme
  save_both(threshold_bar, "GSEA_MSigDB_Hallmark_FDR_threshold_barplot")

  pvalue_bar <- ggplot(plot_data, aes(NES, pathway_label, fill = -log10(nominal_p))) + geom_col(width = 0.65) +
    geom_vline(xintercept = 0, linewidth = 0.4) +
    scale_fill_gradient(low = "#F2E5D5", high = "#8F3B2D", name = expression(-log[10](P))) +
    labs(title = "GSEA colored by nominal P value", x = "Normalized enrichment score", y = NULL) + base_theme
  save_both(pvalue_bar, "GSEA_MSigDB_Hallmark_Pvalue_barplot")

  reference_plot <- ggplot(plot_data, aes(NES, pathway_label, fill = direction)) + geom_col(width = 0.58, color = "black") +
    geom_vline(xintercept = 0, linewidth = 0.5) +
    scale_fill_manual(values = c("CIVA Tumor-high" = "#B5452D", "PDO-high" = "#2C6E91")) +
    labs(title = "GSEA", subtitle = "Positive NES: CIVA Tumor-high; negative NES: PDO-high",
         x = "Normalized enrichment score", y = NULL, fill = NULL) + base_theme +
    theme(plot.title = element_text(face = "bold", hjust = 0.5), legend.position = "top")
  save_both(reference_plot, "GSEA_MSigDB_Hallmark_reference_style")

  significant_sets <- plot_data %>% filter(FDR < 0.05)
  if (nrow(significant_sets) > 0L) {
    top_set <- as.character(significant_sets$pathway[[which.max(abs(significant_sets$NES))]])
    enrichment_plot <- fgsea::plotEnrichment(focused_sets[[top_set]], focused_gene_list) +
      labs(title = unname(display_names[top_set])) + theme_classic(base_size = 12)
    ggsave(file.path(gsea_output_dir, "GSEA_MSigDB_top_Hallmark_enrichment.pdf"), enrichment_plot, width = 9, height = 6)
  }
  print(signed_bar)
}

## 10. Export Hallmark and proliferation scores

`Tumor` in the archived column names means PDO; `CIVA` means CIVA Tumor. To reproduce the original score files, this section intentionally uses the full `combined_tumor` object, matching the source notebook. The corrected pathway analysis above remains cluster-2 excluded.

In [ ]:
if (!requireNamespace("msigdbr", quietly = TRUE)) stop("Install msigdbr to reproduce Hallmark score exports.")
hallmark_rows <- msigdbr_rows("H")
score_set_names <- c(EMT = "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION",
                     Inflamm = "HALLMARK_INFLAMMATORY_RESPONSE",
                     G2M = "HALLMARK_G2M_CHECKPOINT", E2F = "HALLMARK_E2F_TARGETS")
score_source <- combined_tumor
score_sets <- lapply(score_set_names, function(set_name) {
  intersect(unique(as.character(hallmark_rows$gene_symbol[hallmark_rows$gs_name == set_name])), rownames(score_source))
})
if (any(lengths(score_sets) < 2L)) stop("One or more Hallmark score sets has fewer than two available genes.")

group_values <- as.character(score_source$orig.ident)
pad_columns <- function(columns) {
  max_length <- max(lengths(columns))
  as.data.frame(lapply(columns, function(values) c(values, rep(NA_real_, max_length - length(values)))), check.names = FALSE)
}
interleaved_score_columns <- function(score_matrix) {
  list(
    "EMT (Tumor)" = score_matrix["EMT", group_values == "PDO"],
    "EMT (CIVA)" = score_matrix["EMT", group_values == "CIVA Tumor"],
    "Inflamm (Tumor)" = score_matrix["Inflamm", group_values == "PDO"],
    "Inflamm (CIVA)" = score_matrix["Inflamm", group_values == "CIVA Tumor"],
    "G2M (Tumor)" = score_matrix["G2M", group_values == "PDO"],
    "G2M (CIVA)" = score_matrix["G2M", group_values == "CIVA Tumor"],
    "E2F (Tumor)" = score_matrix["E2F", group_values == "PDO"],
    "E2F (CIVA)" = score_matrix["E2F", group_values == "CIVA Tumor"]
  )
}

set.seed(random_seed)
hallmark_scored <- AddModuleScore(score_source, features = score_sets, assay = "RNA", name = "Hallmark_", search = FALSE)
module_matrix <- t(as.matrix(hallmark_scored[[]][, paste0("Hallmark_", seq_along(score_sets)), drop = FALSE]))
rownames(module_matrix) <- names(score_sets)
module_export <- pad_columns(interleaved_score_columns(module_matrix))
write.csv(module_export, file.path(output_dir, "Hallmark_module_scores_Tumor_vs_CIVA.csv"), row.names = FALSE, na = "")

normalized_matrix <- GetAssayData(score_source, assay = "RNA", layer = "data")
direct_matrix <- do.call(rbind, lapply(score_sets, function(genes) Matrix::colMeans(normalized_matrix[genes, , drop = FALSE])))
rownames(direct_matrix) <- names(score_sets)
direct_columns <- interleaved_score_columns(direct_matrix)
direct_export <- pad_columns(direct_columns)
direct_summary <- data.frame(score_group = names(direct_columns),
                             mean_score = vapply(direct_columns, mean, numeric(1)),
                             median_score = vapply(direct_columns, median, numeric(1)), row.names = NULL)
write.csv(direct_export, file.path(output_dir, "Hallmark_direct_scores_Tumor_vs_CIVA.csv"), row.names = FALSE, na = "")
write.csv(direct_summary, file.path(output_dir, "Hallmark_direct_scores_summary_Tumor_vs_CIVA.csv"), row.names = FALSE)

prolif_genes <- intersect(c("MKI67", "TOP2A", "UBE2C", "CENPF", "CDC20", "CDK1", "CCNB1", "CCNB2", "PCNA",
                            "MCM2", "MCM3", "MCM4", "MCM5", "MCM6", "MCM7"), rownames(score_source))
if (length(prolif_genes) == 0L) stop("No proliferation genes are present.")
prolif_scores <- Matrix::colMeans(normalized_matrix[prolif_genes, , drop = FALSE])
prolif_export <- pad_columns(list("Prolif (Tumor)" = prolif_scores[group_values == "PDO"],
                                  "Prolif (CIVA)" = prolif_scores[group_values == "CIVA Tumor"]))
write.csv(prolif_export, file.path(output_dir, "Proliferation_scores_Tumor_vs_CIVA.csv"), row.names = FALSE, na = "")
list(hallmark_direct_summary = direct_summary,
     proliferation_means = colMeans(prolif_export, na.rm = TRUE))

## 11. Validate the reproduced output directory

In [ ]:
expected_top_level <- c(
  "Hallmark_direct_scores_Tumor_vs_CIVA.csv", "Hallmark_direct_scores_summary_Tumor_vs_CIVA.csv",
  "Hallmark_module_scores_Tumor_vs_CIVA.csv", "Proliferation_scores_Tumor_vs_CIVA.csv",
  "cancer_cell_qc.csv", "candidate_sample_metadata.csv", "cell_level_pathway_score_summary.csv",
  "cell_level_pathway_score_tests.csv", "identity_qc_by_group_and_cluster.csv", "pseudobulk_status.csv"
)
expected_gsea <- c(
  "DEG_CIVA_Tumor_vs_PDO_FindAllMarkers.csv", "GSEA_GO_Biological_Process_CIVA_Tumor_vs_PDO.csv",
  "GSEA_GO_prioritized_terms_CIVA_Tumor_vs_PDO.csv", "GSEA_KEGG_CIVA_Tumor_vs_PDO.csv",
  "GSEA_MSigDB_Hallmark_CIVA_Tumor_vs_PDO.csv", "GSEA_GO_Biological_Process_dotplot.pdf",
  "GSEA_top_prioritized_GO_term.pdf", "GSEA_MSigDB_top_Hallmark_enrichment.pdf",
  paste0("GSEA_MSigDB_Hallmark_", c("barplot", "lollipop", "FDR_barplot", "FDR_threshold_barplot", "Pvalue_barplot", "reference_style"), ".pdf"),
  paste0("GSEA_MSigDB_Hallmark_", c("barplot", "lollipop", "FDR_barplot", "FDR_threshold_barplot", "Pvalue_barplot", "reference_style"), ".png")
)
expected_paths <- c(file.path(output_dir, expected_top_level), file.path(gsea_output_dir, expected_gsea))
output_manifest <- data.frame(
  file = sub(paste0("^", output_dir, "/?"), "", expected_paths),
  exists = file.exists(expected_paths),
  bytes = ifelse(file.exists(expected_paths), file.info(expected_paths)$size, NA_real_),
  nonempty = file.exists(expected_paths) & file.info(expected_paths)$size > 0,
  row.names = NULL
)
print(output_manifest)
if (any(!output_manifest$nonempty)) {
  message("Missing/empty files: ", paste(output_manifest$file[!output_manifest$nonempty], collapse = ", "))
  if (!run_expensive_gsea) message("GSEA outputs are generated only when run_expensive_gsea is TRUE.")
} else {
  message("All expected outputs exist and are nonempty.")
}

input_files <- c(combined_tumor = combined_tumor_path)
if (rebuild_integration) input_files <- c(Flex = flex_path, lam = lam_path, input_files)
input_provenance <- data.frame(
  input = names(input_files), path = unname(input_files),
  bytes = unname(file.info(input_files)$size), md5 = unname(tools::md5sum(input_files)), row.names = NULL
)
package_names <- unique(c(core_packages, rebuild_packages, gsea_packages))
package_versions <- data.frame(
  package = package_names,
  version = vapply(package_names, function(package) {
    if (requireNamespace(package, quietly = TRUE)) as.character(packageVersion(package)) else NA_character_
  }, character(1)), row.names = NULL
)
list(input_provenance = input_provenance, package_versions = package_versions, session = sessionInfo())